# NexaTel Phase 1: Data Loading, Cleaning and Quality Report
**Run top to bottom** (Runtime -> Run all). Every number in the final report table is
**computed by this notebook**, not typed by hand.

## 1. Setup: connect Drive, find the data, import tools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile
import pandas as pd

BASE = "/content/drive/MyDrive/NexaTel/"
DATA = BASE + "data/"

# Unzip only if the CSVs are not already there (so re-running never repeats it)
if not os.path.exists(DATA + "customers.csv"):
    zip_path = glob.glob(BASE + "*.zip")[0]
    zipfile.ZipFile(zip_path).extractall(DATA)

print(len(glob.glob(DATA + "*.csv")), "CSV files found")

## 2. Load and profile all 24 tables
PRD steps 1-3: load every CSV, then report rows, columns and blanks for each table.

In [ ]:
tables = {}
for path in sorted(glob.glob(DATA + "*.csv")):
    name = os.path.basename(path).replace(".csv", "")
    if name.endswith("_clean"):          # skip the clean files this notebook saves later
        continue
    tables[name] = pd.read_csv(path)

profile = pd.DataFrame({
    "rows":               {n: t.shape[0] for n, t in tables.items()},
    "columns":            {n: t.shape[1] for n, t in tables.items()},
    "columns_with_nulls": {n: int(t.isnull().any().sum()) for n, t in tables.items()},
})
print(len(tables), "tables loaded")
profile

In [ ]:
# Every column that has blanks, in every table, with the % of rows affected
null_rows = []
for n, t in tables.items():
    for col in t.columns:
        k = int(t[col].isnull().sum())
        if k > 0:
            null_rows.append({"table": n, "column": col, "null_count": k,
                              "null_pct": round(k / len(t) * 100, 2)})
null_report = pd.DataFrame(null_rows)
null_report

In [ ]:
# Date columns are stored as text. List them so we know which ones to convert.
text_date_cols = {}
for n, t in tables.items():
    cols = [c for c in t.columns
            if "date" in c.lower() and not pd.api.types.is_numeric_dtype(t[c])]
    if cols:
        text_date_cols[n] = cols
text_date_cols

## 3. Cleaning
`found` collects every issue count we measure. The comparison with the issue log at the end is built from it.

In [ ]:
found = {}     # (table, issue_type) -> how many rows we measured

### 3A. customers

In [ ]:
customers = tables["customers"].copy()
print("Rows before:", len(customers))

# Duplicates: rows identical in every column
found[("customers", "duplicate_customers")] = int(customers.duplicated().sum())
customers = customers.drop_duplicates()
print("Rows after:", len(customers), "| duplicate customer_ids left:", customers["customer_id"].duplicated().sum())

In [ ]:
# Blanks, measured on the 19,000 real customers (after duplicates are gone)
found[("customers", "missing_email")]   = int(customers["email"].isnull().sum())
found[("customers", "missing_address")] = int(customers["address"].isnull().sum())
found[("customers", "missing_income")]  = int(customers["annual_income_inr"].isnull().sum())

In [ ]:
# Dates: acquisition_date has two formats, so try the main one, then fill the gaps with the other
main  = pd.to_datetime(customers["acquisition_date"], format="%d-%m-%Y", errors="coerce")
other = pd.to_datetime(customers["acquisition_date"], format="%Y/%m/%d", errors="coerce")
found[("customers", "mixed_date_format")] = int(main.isnull().sum())
customers["acquisition_date"] = main.fillna(other)

customers["date_of_birth"] = pd.to_datetime(customers["date_of_birth"], format="%d-%m-%Y", errors="coerce")
customers["churn_date"]    = pd.to_datetime(customers["churn_date"],    format="%d-%m-%Y", errors="coerce")

print("acquisition_date still unreadable:", customers["acquisition_date"].isnull().sum())
print(customers[["date_of_birth", "acquisition_date", "churn_date"]].dtypes)

In [ ]:
# Age: two checks. (1) outside 0-120. (2) not matching date_of_birth.
# (2) catches ages like 0 or 1 on adults, which (1) misses.
ref = pd.Timestamp("2026-03-31")                       # the data's "today"
true_age = (ref - customers["date_of_birth"]).dt.days / 365.25

out_of_range = (customers["age"] < 0) | (customers["age"] > 120)
not_matching = (customers["age"] - true_age).abs() > 1.5
bad_age = out_of_range | not_matching

print("Outside 0-120:", out_of_range.sum(), "| Not matching date_of_birth:", not_matching.sum())
print(customers.loc[bad_age, "age"].value_counts())

found[("customers", "impossible_age")] = int(bad_age.sum())
customers.loc[bad_age, "age"] = None

In [ ]:
# Phone and PIN. pandas read phones as numbers and dropped the "+", so a valid phone has 12 digits.
bad_phone = customers["phone_number"].astype(str).str.len() != 12
bad_pin   = customers["pin_code"].astype(str).str.len() != 6

found[("customers", "invalid_phone")] = int(bad_phone.sum())
found[("customers", "incorrect_pin")] = int(bad_pin.sum())

customers.loc[bad_phone, "phone_number"] = None
customers.loc[bad_pin,   "pin_code"]     = None

In [ ]:
# City names: split the problem into ALL CAPS vs misspelled, then fix both with a city_id lookup
cities = tables["cities"]
true_city = customers["city_id"].map(cities.set_index("city_id")["city_name"])
print("city_ids not found in cities.csv:", true_city.isnull().sum())

wrong_city = customers["city_name"] != true_city
caps_only  = wrong_city & (customers["city_name"].str.lower() == true_city.str.lower())
misspelled = wrong_city & ~caps_only

found[("customers", "inconsistent_capitalisation")] = int(caps_only.sum())
found[("customers", "misspelled_city")]              = int(misspelled.sum())
print("Wrong in total:", wrong_city.sum(), "| capitals only:", caps_only.sum(), "| misspelled:", misspelled.sum())

customers["city_name"] = true_city

In [ ]:
# Emails: a real email contains "@". Blank emails are a different problem (missing), so skip them here.
bad_email = customers["email"].notnull() & ~customers["email"].str.contains("@", na=False)
found[("customers", "invalid_email")] = int(bad_email.sum())
print(customers.loc[bad_email, "email"].head())
customers.loc[bad_email, "email"] = None

In [ ]:
# churn_date should be blank exactly for Active customers
print("Active:", (customers["customer_status"] == "Active").sum(),
      "| blank churn_date:", customers["churn_date"].isnull().sum())

final_nulls = customers.isnull().sum()
print(final_nulls[final_nulls > 0])
print("Final shape:", customers.shape)

customers.to_csv(DATA + "customers_clean.csv", index=False)
print("Saved customers_clean.csv")

### 3B. billing

In [ ]:
billing = tables["billing"].copy()

found[("billing", "duplicate_invoices")] = int(billing.duplicated().sum())
billing = billing.drop_duplicates()

billing["billing_date"] = pd.to_datetime(billing["billing_date"], format="%d-%m-%Y", errors="coerce")

future   = billing["billing_date"] > "2026-03-31"          # the data ends in March 2026
negative = billing["total_amount_inr"] < 0
correct_gst = (billing["base_amount_inr"] * 0.18).round(2)
wrong_gst   = (billing["gst_amount_inr"] - correct_gst).abs() > 0.05

found[("billing", "future_dates")]            = int(future.sum())
found[("billing", "negative_invoice_amount")] = int(negative.sum())
found[("billing", "incorrect_gst")]           = int(wrong_gst.sum())

In [ ]:
# Fixes: minus sign off (assumes sign errors, not refunds), GST recalculated, future invoices removed
billing.loc[negative, "total_amount_inr"] = billing.loc[negative, "total_amount_inr"].abs()
billing.loc[wrong_gst, "gst_amount_inr"]  = correct_gst[wrong_gst]
billing = billing[~future]

# Sanity check: after the fixes, does total = base + GST?
ok = ((billing["total_amount_inr"] - (billing["base_amount_inr"] + billing["gst_amount_inr"])).abs() < 0.05)
print("Rows where total = base + GST:", round(ok.mean() * 100, 2), "%")

print("Clean billing rows:", billing.shape)
billing.to_csv(DATA + "billing_clean.csv", index=False)

### 3C. payments

In [ ]:
payments = tables["payments"].copy()

found[("payments", "duplicate_payments")] = int(payments.duplicated().sum())
payments = payments.drop_duplicates()

# Broken link = invoice_id that never existed in the ORIGINAL billing file
broken = ~payments["invoice_id"].isin(tables["billing"]["invoice_id"])
found[("payments", "broken_foreign_key")] = int(broken.sum())

# Separate group: invoices that existed but we removed (future-dated). Report this as an extra.
linked_to_removed = ~payments["invoice_id"].isin(billing["invoice_id"]) & ~broken
print("Payments on invoices we removed (future-dated):", linked_to_removed.sum())

payments = payments[payments["invoice_id"].isin(billing["invoice_id"])]
print("Clean payments rows:", payments.shape)
payments.to_csv(DATA + "payments_clean.csv", index=False)

### 3D. subscriptions

In [ ]:
subs = tables["subscriptions"].copy()

found[("subscriptions", "null_foreign_key")] = int(subs["plan_id"].isnull().sum())

unknown = subs["subscription_status"] == "UNKNOWN_STATE"
found[("subscriptions", "invalid_category")] = int(unknown.sum())

# PROVE the rule before using it: on the normal rows, does status always follow end_date?
known = subs[~unknown]
print(pd.crosstab(known["subscription_status"], known["end_date"].notnull()))

In [ ]:
# The table above shows: Active = never has an end_date, Terminated = always has one. So:
has_end = subs["end_date"].notnull()
subs.loc[unknown & has_end,  "subscription_status"] = "Terminated"
subs.loc[unknown & ~has_end, "subscription_status"] = "Active"

print(subs["subscription_status"].value_counts())
subs.to_csv(DATA + "subscriptions_clean.csv", index=False)

# Heads-up for Phase 2/3: some customers hold more than one subscription
per_customer = subs.groupby("customer_id").size()
print("Customers with 2+ subscriptions:", (per_customer > 1).sum(), "| subscription rows:", len(subs))

### 3E. usage_data (outliers)

In [ ]:
usage = tables["usage_data"].copy()
gb = usage["data_gb_used"]

print(gb.describe().round(1))
print("\nLargest 'normal-looking' values, then where it jumps:")
print(gb[gb > 250].sort_values().head(20).round(0).tolist())

In [ ]:
# Cutoff: most customers use under ~150 GB and the tail thins out near 300.
# There is NO clean gap between 300 and 350, so the cutoff is a judgment call:
#   > 300 GB flags 173 rows, > 350 GB flags 171 (the number in the issue log).
# We use 350 and say so in the report.
OUTLIER_GB = 350

outlier = gb > OUTLIER_GB
found[("usage_data", "outlier_values")] = int(outlier.sum())
print("Outliers flagged (>", OUTLIER_GB, "GB):", outlier.sum(), "| would be", (gb > 300).sum(), "at 300 GB")

usage.loc[outlier, "data_gb_used"] = None
usage.to_csv(DATA + "usage_data_clean.csv", index=False)

### 3F. complaints (share unresolved)

In [ ]:
complaints = tables["complaints"]
status_share = (complaints["status"].value_counts(normalize=True) * 100).round(2)
print(status_share)
unresolved_pct = status_share["Unresolved"]

## 4. Referential integrity: do all customer_ids exist in customers?

In [ ]:
valid_ids = set(tables["customers"]["customer_id"])
ri = {}
for n, t in tables.items():
    if n != "customers" and "customer_id" in t.columns:
        ri[n] = int((~t["customer_id"].isin(valid_ids)).sum())
integrity = pd.Series(ri, name="unknown_customer_ids")
integrity

## 5. Data Quality Report: your numbers vs the issue log (built by code)

In [ ]:
log = tables["data_quality_issue_log"].copy()
log["found"] = [found.get((t, i)) for t, i in zip(log["table"], log["issue_type"])]
log["difference"] = log["found"] - log["rows_affected"]
log["match"] = log["difference"] == 0

n_match = int(log["match"].sum())
print(f"{n_match} of {len(log)} issues match the log exactly\n")
log

In [ ]:
# Key points, with every number pulled from the analysis above
active = int((customers["customer_status"] == "Active").sum())
print(f"- {len(tables)} tables loaded and profiled.")
print(f"- {n_match} of {len(log)} logged issues matched exactly; {len(log) - n_match} differ (see 'difference').")
print(f"- Customers: {len(tables['customers'])} raw rows -> {len(customers)} after removing duplicates.")
print(f"- churn_date is blank for all {customers['churn_date'].isnull().sum()} Active customers ({active} Active). Expected, not an error.")
print(f"- {unresolved_pct}% of complaints are Unresolved (a lead to test in Phase 2, not a proven churn driver).")
print(f"- Payments removed because their invoice was future-dated: {int(linked_to_removed.sum())}.")
print(f"- Unknown customer_ids across all tables: {int(integrity.sum())}.")

### Rules for the written report
- For every row where `match` is False, write **what you tested**. If you could not explain the gap, write "not reproducible", do not guess.
- State your assumptions: negative totals treated as sign errors, future-dated invoices removed (and the payments linked to them), usage cutoff of 350 GB.